# ETRS923 – TP3 : classification météo (Meteostat → TensorFlow → ONNX / TFLite)

Ce notebook couvre la partie 1 du TP (entraînement + étude du compromis classes / taille / précision)
et prépare TOUS les fichiers nécessaires aux parties 2 (ThingSpeak/MATLAB) et 3 (STM32N6) :

- `meteo_model.onnx` : modèle float32 pour MATLAB et X-CUBE-AI (Cortex-M55)
- `meteo_model_int8.tflite` : modèle INT8 pour le NPU Neural-ART (option)
- `meteo_model_params.h` : moyennes, écarts-types, noms de classes, paramètres INT8 → à copier dans STM32CubeIDE
- `meteo_test_vectors.h` / `.csv` : entrées + sorties attendues pour valider la carte et MATLAB
- `meteo_weights.json` / `meteo_weights_inline.m` : poids du réseau pour l'inférence ThingSpeak sans toolbox

Exécuter les cellules dans l'ordre (Exécution → Tout exécuter).

In [ ]:
# 0. Installation (Colab a déjà TensorFlow, numpy, pandas, sklearn)
!pip install -q "meteostat>=2.0" tf2onnx onnxruntime

In [ ]:
import os, json, time, zipfile
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
keras.utils.set_random_seed(SEED)          # résultats reproductibles
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

OUT = "export"                             # dossier de sortie (zippé à la fin)
os.makedirs(OUT, exist_ok=True)

## 1. Téléchargement des données Meteostat
On prend les stations à moins de 120 km du campus du Bourget-du-Lac (Savoie, Isère, Haute-Savoie, Lyon, Genève…),
sur 10 ans. Le résultat est mis en cache dans `meteo_raw.parquet` : relancer le notebook ne re-télécharge rien.

In [ ]:
CENTRE = (45.64, 5.87)        # campus Savoie Technolac, Le Bourget-du-Lac (lat, lon)
STATION_LOCALE = "07491"      # Chambéry-Aix-les-Bains (aéroport, à côté du campus, 235 m) : toujours incluse
RAYON_M = 120_000             # 120 km
NB_STATIONS_MAX = 12
DEBUT, FIN = datetime(2016, 1, 1), datetime(2025, 12, 31, 23, 59)
RAW = "meteo_raw.parquet"

def telecharger():
    import meteostat as ms
    proches = ms.stations.nearby(ms.Point(*CENTRE), radius=RAYON_M, limit=40)
    print(f"{len(proches)} stations dans le rayon ; station locale présente : {STATION_LOCALE in proches.index}")
    morceaux = []
    for sid, info in proches.iterrows():
        try:
            df = ms.hourly(ms.Station(id=sid), DEBUT, FIN).fetch()
        except Exception as e:
            print("  ", sid, "erreur", e); continue
        if df is None or "coco" not in df or df["coco"].notna().mean() < 0.5:
            continue                                   # station sans code météo exploitable
        df = df.reset_index()[["time", "temp", "rhum", "pres", "coco"]]
        df["station"] = sid
        morceaux.append(df)
        print(f"  {sid} {info['name']:<30} {len(df):>7} lignes, coco dispo {df['coco'].notna().mean():.0%}")
        if len(morceaux) >= NB_STATIONS_MAX:
            break
    out = pd.concat(morceaux, ignore_index=True)
    out.to_parquet(RAW, index=False)
    return out

raw = pd.read_parquet(RAW) if os.path.exists(RAW) else telecharger()
raw["time"] = pd.to_datetime(raw["time"])
print(raw.shape); raw.head()

## 2. Variables d'entrée (features)
Ce que la carte X-NUCLEO-IKS01A3 sait mesurer : température (HTS221), humidité (HTS221), pression (LPS22HH).

- `pres` de Meteostat = pression **ramenée au niveau de la mer**. La carte mesure la pression **à son altitude**
  → il faudra corriger côté STM32 et côté MATLAB (formule dans le guide).
- `dp3h` = variation de pression sur 3 h (tendance barométrique). C'est l'indicateur historique des baromètres :
  pression qui chute = dégradation. Gros avantage : la pression est la même dedans et dehors, donc cette feature
  reste valable même si la carte est dans la salle de TP.
- `spread` = écart température – point de rosée (calculé à partir de T et HR). Proche de 0 → air saturé (brouillard, pluie).
- `hsin/hcos` = heure de la journée (nécessite l'heure sur la carte : SNTP/RTC).

In [ ]:
def point_de_rosee(T, RH):
    """Formule de Magnus (°C). Même formule à recopier en C et en MATLAB."""
    a, b = 17.62, 243.12
    g = np.log(np.clip(RH, 1, 100) / 100.0) + a * T / (b + T)
    return b * g / (a - g)

def ajouter_features(df):
    df = df.sort_values(["station", "time"]).copy()
    morceaux = []
    for sid, d in df.groupby("station"):
        d = d.set_index("time").asfreq("h")          # grille horaire régulière (trous = NaN)
        d["station"] = sid
        d["dp3h"] = d["pres"] - d["pres"].shift(3)   # tendance sur 3 h
        morceaux.append(d.reset_index())
    df = pd.concat(morceaux, ignore_index=True)
    df["spread"] = df["temp"] - point_de_rosee(df["temp"], df["rhum"])
    h = df["time"].dt.hour
    df["hsin"], df["hcos"] = np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)
    return df

data = ajouter_features(raw)
data[["temp", "rhum", "pres", "dp3h", "spread"]].describe().round(2)

## 3. Classes : codes Meteostat (`coco`) → classes du TP
Codes Meteostat : 1 Clair, 2 Peu nuageux, 3 Nuageux, 4 Couvert, 5 Brouillard, 6 Brouillard givrant,
7 Pluie faible, 8 Pluie, 9 Forte pluie, 10 Pluie verglaçante, 11 Forte pluie verglaçante, 12 Pluie et neige,
13 Forte pluie et neige, 14 Neige faible, 15 Neige, 16 Forte neige, 17 Averse, 18 Forte averse,
19 Averse pluie+neige, 20 Forte averse pluie+neige, 21 Averse de neige, 22 Forte averse de neige,
23 Éclairs, 24 Grêle, 25 Orage, 26 Orage violent, 27 Tempête.

On définit plusieurs **schémas de classes** pour l'étude du compromis (question 1.b).

In [ ]:
SCHEMAS = {
    "K2": {"Sec": [1, 2, 3, 4, 5, 6], "Precipitations": list(range(7, 28))},
    "K3": {"Clair": [1, 2], "Nuageux": [3, 4], "Precipitations": list(range(7, 28))},
    "K4": {"Clair": [1, 2], "Nuageux": [3, 4],
           "Pluie": [7, 8, 9, 10, 11, 17, 18, 23, 24, 25, 26, 27],
           "Neige": [12, 13, 14, 15, 16, 19, 20, 21, 22]},
    "K5": {"Clair": [1, 2], "Nuageux": [3, 4], "Brouillard": [5, 6],
           "Pluie": [7, 8, 9, 10, 11, 17, 18, 23, 24, 25, 26, 27],
           "Neige": [12, 13, 14, 15, 16, 19, 20, 21, 22]},
    "K8": {"Clair": [1], "Peu nuageux": [2], "Nuageux": [3], "Couvert": [4], "Brouillard": [5, 6],
           "Pluie": [7, 8, 9, 10, 11, 17, 18], "Neige": [12, 13, 14, 15, 16, 19, 20, 21, 22],
           "Orage": [23, 24, 25, 26, 27]},
}

def etiqueter(df, schema):
    table = {code: i for i, codes in enumerate(SCHEMAS[schema].values()) for code in codes}
    y = df["coco"].map(table)
    return y, list(SCHEMAS[schema].keys())

for nom in SCHEMAS:
    y, noms = etiqueter(data, nom)
    rep = y.value_counts(normalize=True).sort_index()
    print(nom, {noms[int(k)]: f"{v:.1%}" for k, v in rep.items()})

## 4. Découpage temporel train / validation / test
On NE mélange PAS les heures au hasard : deux heures consécutives sont presque identiques, un découpage aléatoire
mettrait des « jumeaux » dans train et test et gonflerait artificiellement l'accuracy.
Train = 2016–2023, validation = 2024, test = 2025 (année jamais vue).

In [ ]:
FEATURE_SETS = {
    "P_seule (robuste en intérieur)": ["pres", "dp3h"],
    "T_HR_P": ["temp", "rhum", "pres"],
    "T_HR_P_dP": ["temp", "rhum", "pres", "dp3h"],
    "T_HR_P_dP_spread_heure": ["temp", "rhum", "pres", "dp3h", "spread", "hsin", "hcos"],
}

def preparer(features, schema):
    y, noms = etiqueter(data, schema)
    d = data.assign(y=y).dropna(subset=features + ["y"])
    an = d["time"].dt.year
    parts = {}
    for nom, masque in [("train", an <= 2023), ("val", an == 2024), ("test", an == 2025)]:
        parts[nom] = (d.loc[masque, features].to_numpy(np.float32), d.loc[masque, "y"].to_numpy(np.int32))
    mean = parts["train"][0].mean(axis=0)
    std = parts["train"][0].std(axis=0) + 1e-6
    norm = {k: ((X - mean) / std, y) for k, (X, y) in parts.items()}
    return norm, mean, std, noms

## 5. Modèle et entraînement

In [ ]:
ARCHIS = {"XS 8": [8], "S 16-8": [16, 8], "M 32-16": [32, 16], "L 64-32-16": [64, 32, 16]}

def construire(n_in, n_out, couches, dropout=0.1):
    model = keras.Sequential([keras.layers.Input(shape=(n_in,), name="input")])
    for n in couches:
        model.add(keras.layers.Dense(n, activation="relu"))
        if dropout:
            model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(n_out, activation="softmax"))
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def poids_classes(y, n):
    # 'balanced' adouci par une racine : aide les classes rares sans écraser la classe majoritaire
    presentes = np.unique(y)
    w = compute_class_weight("balanced", classes=presentes, y=y)
    poids = {i: 1.0 for i in range(n)}                       # classe absente du train : poids neutre
    poids.update({int(c): float(np.sqrt(v)) for c, v in zip(presentes, w)})
    return poids

def entrainer(parts, n_out, couches, epochs=60, verbose=0):
    (Xtr, ytr), (Xva, yva) = parts["train"], parts["val"]
    model = construire(Xtr.shape[1], n_out, couches)
    hist = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs, batch_size=512,
                     class_weight=poids_classes(ytr, n_out), verbose=verbose,
                     callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
                                keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)])
    return model, hist

def evaluer(model, parts, n_out):
    Xte, yte = parts["test"]
    pred = model.predict(Xte, batch_size=4096, verbose=0).argmax(axis=1)
    n_params = model.count_params()
    return {"accuracy": accuracy_score(yte, pred),
            "balanced_acc": balanced_accuracy_score(yte, pred),
            "macro_F1": f1_score(yte, pred, average="macro", labels=np.arange(n_out), zero_division=0),
            "baseline_majoritaire": float(np.bincount(yte, minlength=n_out).max() / len(yte)),
            "params": n_params, "float32_Ko": n_params * 4 / 1024, "int8_Ko": n_params / 1024}

## 6. Étude du compromis (question 1.b)
Trois balayages, chacun en faisant varier UN seul facteur :
1. nombre de classes (features et architecture fixes)
2. taille du modèle (classes et features fixes)
3. jeu de features (classes et architecture fixes)

Lire **macro_F1 / balanced_acc** plutôt que l'accuracy : avec des classes déséquilibrées,
un modèle qui répond toujours « Nuageux » a déjà une accuracy élevée (voir `baseline_majoritaire`).

In [ ]:
def etude(liste):
    lignes = []
    for schema, fset, archi in liste:
        parts, _, _, noms = preparer(FEATURE_SETS[fset], schema)
        t0 = time.time()
        model, _ = entrainer(parts, len(noms), ARCHIS[archi])
        r = evaluer(model, parts, len(noms))
        r.update({"classes": schema, "features": fset, "archi": archi, "duree_s": round(time.time() - t0, 1)})
        lignes.append(r); print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()})
    cols = ["classes", "features", "archi", "params", "float32_Ko", "int8_Ko",
            "accuracy", "balanced_acc", "macro_F1", "baseline_majoritaire", "duree_s"]
    return pd.DataFrame(lignes)[cols].round(3)

FSET_REF, ARCHI_REF, SCHEMA_REF = "T_HR_P_dP", "M 32-16", "K4"
etude_classes = etude([(s, FSET_REF, ARCHI_REF) for s in SCHEMAS])
etude_taille = etude([(SCHEMA_REF, FSET_REF, a) for a in ARCHIS])
etude_features = etude([(SCHEMA_REF, f, ARCHI_REF) for f in FEATURE_SETS])
display(etude_classes); display(etude_taille); display(etude_features)
pd.concat([etude_classes, etude_taille, etude_features]).to_csv(f"{OUT}/etude_compromis.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot([len(SCHEMAS[s]) for s in etude_classes["classes"]], etude_classes["macro_F1"], "o-", label="macro-F1")
ax[0].plot([len(SCHEMAS[s]) for s in etude_classes["classes"]], etude_classes["accuracy"], "s--", label="accuracy")
ax[0].set(xlabel="nombre de classes", title="Effet du nombre de classes"); ax[0].legend()
ax[1].semilogx(etude_taille["params"], etude_taille["macro_F1"], "o-")
for _, r in etude_taille.iterrows(): ax[1].annotate(r["archi"], (r["params"], r["macro_F1"]))
ax[1].set(xlabel="nombre de paramètres (log)", ylabel="macro-F1", title="Effet de la taille")
ax[2].barh(etude_features["features"], etude_features["macro_F1"]); ax[2].set(xlabel="macro-F1", title="Effet des features")
plt.tight_layout(); plt.savefig(f"{OUT}/etude_compromis.png", dpi=120); plt.show()

## 7. Modèle final
Choisir la configuration à partir de l'étude (le meilleur compromis, pas forcément le meilleur score).
Par défaut : 4 classes, features T/HR/P/ΔP3h, réseau 32-16.

In [ ]:
SCHEMA, FEATURES, COUCHES = "K4", FEATURE_SETS["T_HR_P_dP"], ARCHIS["M 32-16"]

parts, MEAN, STD, CLASSES = preparer(FEATURES, SCHEMA)
N_IN, N_OUT = len(FEATURES), len(CLASSES)
model, hist = entrainer(parts, N_OUT, COUCHES, epochs=100, verbose=2)
model.summary()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val"); ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="val"); ax[1].set_title("accuracy"); ax[1].legend()
plt.savefig(f"{OUT}/courbes_apprentissage.png", dpi=120); plt.show()

Xte, yte = parts["test"]
t0 = time.perf_counter()
proba_keras = model.predict(Xte, batch_size=4096, verbose=0)
t_python_us = (time.perf_counter() - t0) / len(Xte) * 1e6
pred = proba_keras.argmax(axis=1)
print(classification_report(yte, pred, labels=np.arange(N_OUT), target_names=CLASSES, digits=3, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(yte, pred, labels=np.arange(N_OUT), normalize="true"), display_labels=CLASSES).plot(values_format=".2f")
plt.title("Matrice de confusion (test 2025, normalisée par ligne)"); plt.savefig(f"{OUT}/confusion.png", dpi=120); plt.show()

# temps d'inférence pour UN échantillon (comparaison Cloud/Edge)
x1 = Xte[:1]
for _ in range(10): model(x1, training=False)
t0 = time.perf_counter()
for _ in range(200): model(x1, training=False)
print(f"Inférence Keras 1 échantillon : {(time.perf_counter() - t0) / 200 * 1e6:.0f} µs (Colab CPU)")
model.save(f"{OUT}/meteo_model.keras")

## 8. Export ONNX (pour MATLAB et X-CUBE-AI)
Attention : l'exemple du sujet (`tf2onnx.convert.from_keras`) **plante avec Keras 3** (version actuelle de Colab).
On passe par une `tf.function`, ce qui donne le même graphe ONNX : MatMul → Add → Relu → … → Softmax.
Entrée fixée à (1, N_IN) : une mesure à la fois, c'est exactement l'usage sur ThingSpeak et sur la carte,
et X-CUBE-AI / MATLAB n'ont pas de dimension inconnue à deviner.

In [ ]:
import tf2onnx, onnx, onnxruntime as ort

signature = [tf.TensorSpec((1, N_IN), tf.float32, name="input")]
@tf.function(input_signature=signature)
def inference(x):
    return {"probs": model(x, training=False)}

tf2onnx.convert.from_function(inference, input_signature=signature, opset=13,
                              output_path=f"{OUT}/meteo_model.onnx")
onx = onnx.load(f"{OUT}/meteo_model.onnx"); onnx.checker.check_model(onx)
print("Opérateurs ONNX :", [n.op_type for n in onx.graph.node])

sess = ort.InferenceSession(f"{OUT}/meteo_model.onnx")
print("entrée :", sess.get_inputs()[0].name, sess.get_inputs()[0].shape, "| sortie :", sess.get_outputs()[0].name)
proba_onnx = np.vstack([sess.run(None, {"input": Xte[i:i+1]})[0] for i in range(500)])
print("Écart max Keras / ONNX :", np.abs(proba_onnx - proba_keras[:500]).max())   # doit être ~1e-6
t0 = time.perf_counter()
for i in range(1000): sess.run(None, {"input": Xte[i:i+1]})
t_onnx_us = (time.perf_counter() - t0) / 1000 * 1e6
print(f"Inférence ONNX Runtime 1 échantillon : {t_onnx_us:.1f} µs (référence 'calcul pur' côté serveur)")

## 9. Export TFLite float et INT8 (pour l'option NPU Neural-ART)
Le NPU ne calcule qu'en entiers 8 bits. La quantification « post-training » utilise des données
représentatives (ici la validation) pour choisir l'échelle de chaque tenseur.

In [ ]:
def representatif():
    Xva = parts["val"][0]
    idx = np.random.default_rng(0).choice(len(Xva), size=min(1000, len(Xva)), replace=False)
    for i in idx:
        yield [Xva[i:i+1]]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
open(f"{OUT}/meteo_model_f32.tflite", "wb").write(conv.convert())

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representatif
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl_int8 = conv.convert()
open(f"{OUT}/meteo_model_int8.tflite", "wb").write(tfl_int8)

interp = tf.lite.Interpreter(model_content=tfl_int8); interp.allocate_tensors()
din, dout = interp.get_input_details()[0], interp.get_output_details()[0]
IN_SCALE, IN_ZP = din["quantization"]; OUT_SCALE, OUT_ZP = dout["quantization"]
print(f"INT8 : {len(tfl_int8)} octets | in scale={IN_SCALE:.6f} zp={IN_ZP} | out scale={OUT_SCALE:.6f} zp={OUT_ZP}")

def predire_int8(X):
    preds = []
    for x in X:
        q = np.clip(np.round(x / IN_SCALE + IN_ZP), -128, 127).astype(np.int8)[None, :]
        interp.set_tensor(din["index"], q); interp.invoke()
        preds.append(interp.get_tensor(dout["index"])[0].argmax())
    return np.array(preds)

n_eval = min(5000, len(Xte))
acc_int8 = accuracy_score(yte[:n_eval], predire_int8(Xte[:n_eval]))
acc_f32 = accuracy_score(yte[:n_eval], pred[:n_eval])
print(f"Accuracy float32 = {acc_f32:.4f} | INT8 = {acc_int8:.4f} | perte = {(acc_f32 - acc_int8) * 100:.2f} points")

## 10. Fichiers pour MATLAB / ThingSpeak et pour le STM32
Tout est généré automatiquement depuis les valeurs réelles de l'entraînement :
aucun copier-coller manuel de moyennes ou d'échelles (source classique de bugs).

In [ ]:
# 10.a Poids en JSON (inférence ThingSpeak sans Deep Learning Toolbox) + vérification numpy
denses = [l for l in model.layers if isinstance(l, keras.layers.Dense)]
poids = {"features": FEATURES, "classes": CLASSES, "mean": MEAN.tolist(), "std": STD.tolist(),
         "W": [l.get_weights()[0].T.tolist() for l in denses],     # W{k} : (sorties x entrées), convention MATLAB y = W*x + b
         "b": [l.get_weights()[1].tolist() for l in denses]}
json.dump(poids, open(f"{OUT}/meteo_weights.json", "w"))

def forward_numpy(xn):
    h = xn
    for k, (W, b) in enumerate(zip(poids["W"], poids["b"])):
        h = np.asarray(W) @ h + np.asarray(b)
        if k < len(denses) - 1: h = np.maximum(h, 0)
    e = np.exp(h - h.max()); return e / e.sum()
print("Écart max Keras / numpy :", max(np.abs(forward_numpy(Xte[i]) - proba_keras[i]).max() for i in range(200)))

# 10.a bis Même chose sous forme de code MATLAB à coller dans l'app MATLAB Analysis de ThingSpeak
def m_mat(M):
    M = np.atleast_2d(np.asarray(M))
    return "[" + "; ".join(" ".join(f"{v:.7g}" for v in ligne) for ligne in M) + "]"
m = ["% Généré par TP3_Meteo_Colab.ipynb - coller dans la section POIDS du script ThingSpeak"]
m.append("P.features = {" + ", ".join(f"'{f}'" for f in FEATURES) + "};")
m.append("P.classes = {" + ", ".join(f"'{c}'" for c in CLASSES) + "};")
m.append(f"P.mean = {m_mat(MEAN)}';")
m.append(f"P.std  = {m_mat(STD)}';")
m.append("P.W = cell(1, %d); P.b = cell(1, %d);" % (len(denses), len(denses)))
for k, (W, b) in enumerate(zip(poids["W"], poids["b"])):
    m.append(f"P.W{{{k+1}}} = {m_mat(W)};")
    m.append(f"P.b{{{k+1}}} = {m_mat(b)}';")
open(f"{OUT}/meteo_weights_inline.m", "w").write("\n".join(m))
print("meteo_weights_inline.m :", sum(len(l) for l in m), "caractères")

# 10.b Vecteurs de test : valeurs PHYSIQUES (non normalisées) + probabilités attendues
rng = np.random.default_rng(1)
idx = [rng.choice(np.where(yte == c)[0]) for c in range(N_OUT) if (yte == c).any()][:N_OUT] + list(rng.choice(len(Xte), 4, replace=False))
X_phys = Xte[idx] * STD + MEAN
P_ref = proba_keras[idx]
pd.DataFrame(np.hstack([X_phys, P_ref]), columns=FEATURES + [f"p_{c}" for c in CLASSES]).round(5) \
  .assign(classe_attendue=[CLASSES[i] for i in P_ref.argmax(1)]).to_csv(f"{OUT}/meteo_test_vectors.csv", index=False)

# 10.c En-tête C pour STM32CubeIDE
def c_tab(v): return "{ " + ", ".join(f"{x:.6f}f" for x in v) + " }"
h = []
h.append("/* Généré automatiquement par TP3_Meteo_Colab.ipynb le " + datetime.now().strftime("%Y-%m-%d %H:%M") + " - ne pas modifier à la main */")
h.append("#ifndef METEO_MODEL_PARAMS_H\n#define METEO_MODEL_PARAMS_H\n")
h.append(f"#define METEO_NB_FEATURES {N_IN}   /* ordre : {', '.join(FEATURES)} */")
h.append(f"#define METEO_NB_CLASSES  {N_OUT}\n")
h.append(f"static const float METEO_MEAN[METEO_NB_FEATURES] = {c_tab(MEAN)};")
h.append(f"static const float METEO_STD[METEO_NB_FEATURES]  = {c_tab(STD)};")
h.append("static const char *const METEO_CLASS_NAMES[METEO_NB_CLASSES] = { " + ", ".join(f'"{c}"' for c in CLASSES) + " };\n")
h.append("/* Paramètres de quantification INT8 (uniquement pour l'option NPU) */")
h.append(f"#define METEO_IN_SCALE  {IN_SCALE:.9f}f\n#define METEO_IN_ZP     ({int(IN_ZP)})")
h.append(f"#define METEO_OUT_SCALE {OUT_SCALE:.9f}f\n#define METEO_OUT_ZP    ({int(OUT_ZP)})\n")
h.append(f"#define METEO_NB_TESTS {len(idx)}")
h.append("static const float METEO_TEST_IN[METEO_NB_TESTS][METEO_NB_FEATURES] = {")
h += ["  " + c_tab(x) + "," for x in X_phys]
h.append("};\nstatic const float METEO_TEST_OUT[METEO_NB_TESTS][METEO_NB_CLASSES] = {")
h += ["  " + c_tab(p) + "," for p in P_ref]
h.append("};\n\n#endif")
open(f"{OUT}/meteo_model_params.h", "w").write("\n".join(h))
print("\n".join(h[:12]))

# 10.d Résumé chiffré pour le rapport
resume = {"schema": SCHEMA, "classes": CLASSES, "features": FEATURES, "couches": COUCHES, "params": model.count_params(),
          "test_accuracy_f32": float(accuracy_score(yte, pred)), "test_macroF1": float(f1_score(yte, pred, average="macro", labels=np.arange(N_OUT), zero_division=0)),
          "test_accuracy_int8_5000": float(acc_int8), "int8_tflite_octets": len(tfl_int8),
          "inference_python_batch_us_par_echantillon": t_python_us, "inference_onnxruntime_1_echantillon_us": t_onnx_us}
json.dump(resume, open(f"{OUT}/resume.json", "w"), indent=2, ensure_ascii=False); print(resume)

## 12. (À faire après la partie 3) Comparaison Edge / Cloud / météo réellement observée
Les deux solutions exécutent le MÊME réseau : leur désaccord ne vient que des entrées
(moyenne sur 5 min côté Cloud, arrondis, décalage temporel). La vraie « accuracy » se mesure
contre l'observation officielle de la station Meteostat la plus proche, heure par heure.
Remplir les identifiants puis exécuter la cellule.

In [ ]:
CANAL_MESURES, CLE_MESURES = 0, ""        # canal TP2 (la carte y écrit aussi ses résultats Edge)
CHAMP_EDGE_CLASSE, CHAMP_EDGE_US = 4, 6   # champs où la carte écrit sa classe et son temps d'inférence
CANAL_CLOUD, CLE_CLOUD = 0, ""            # canal résultat écrit par ThingSpeak (field1 = classe, field3 = ms, field4 = âge s)
STATION_REF = STATION_LOCALE              # 07491 Chambéry-Aix, à côté du campus du Bourget-du-Lac

def lire_thingspeak(canal, cle, n=8000):
    url = f"https://api.thingspeak.com/channels/{canal}/feeds.csv?results={n}" + (f"&api_key={cle}" if cle else "")
    df = pd.read_csv(url)
    df["created_at"] = pd.to_datetime(df["created_at"].str.replace(" UTC", "", regex=False), utc=True).dt.tz_localize(None)
    return df.sort_values("created_at")

def comparer(edge, cloud, obs=None):
    e = edge[["created_at", f"field{CHAMP_EDGE_CLASSE}", f"field{CHAMP_EDGE_US}"]].dropna()
    e.columns = ["t", "classe_edge", "us_edge"]
    c = cloud[["created_at", "field1", "field3", "field4"]].dropna()
    c.columns = ["t", "classe_cloud", "ms_cloud", "age_cloud_s"]
    m = pd.merge_asof(c, e, on="t", direction="backward", tolerance=pd.Timedelta("2min")).dropna()
    print(f"{len(m)} couples Edge/Cloud | accord = {(m.classe_edge == m.classe_cloud).mean():.1%}")
    print(f"Edge : inférence médiane {m.us_edge.median():.0f} µs (calcul seul, sur la carte)")
    print(f"Cloud : inférence médiane {m.ms_cloud.median()*1000:.0f} µs (calcul seul) | "
          f"délai mesure → résultat médian {m.age_cloud_s.median():.0f} s (réseau + ThingSpeak + React)")
    if obs is not None:
        m["heure"] = m["t"].dt.floor("h")
        o = obs.rename(columns={"time": "heure"})[["heure", "y_obs"]]
        m = m.merge(o, on="heure", how="inner").dropna()
        print(f"{len(m)} heures avec observation | accuracy Edge = {(m.classe_edge == m.y_obs).mean():.1%} "
              f"| accuracy Cloud = {(m.classe_cloud == m.y_obs).mean():.1%}")
    return m

if CANAL_MESURES and CANAL_CLOUD:
    edge, cloud = lire_thingspeak(CANAL_MESURES, CLE_MESURES), lire_thingspeak(CANAL_CLOUD, CLE_CLOUD)
    obs = None
    if STATION_REF:
        import meteostat as ms
        o = ms.hourly(ms.Station(id=STATION_REF), edge.created_at.min().floor("h").to_pydatetime(),
                      edge.created_at.max().ceil("h").to_pydatetime()).fetch()
        if o is not None:
            o = o.reset_index()
            o["y_obs"] = etiqueter(o, SCHEMA)[0]
            obs = o
    resultat = comparer(edge, cloud, obs)
    resultat.to_csv(f"{OUT}/comparaison_edge_cloud.csv", index=False)

In [ ]:
# 11. Tout récupérer dans un zip
with zipfile.ZipFile("TP3_export.zip", "w") as z:
    for f in os.listdir(OUT):
        z.write(os.path.join(OUT, f), f)
print(sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download("TP3_export.zip")
except ImportError:
    pass